# RajeshPradhan Business Intelligence Data Analytics

## Business Intelligence & Data Analytics Project

**Objective:** Transform business data into actionable management insights using the BI flow demonstrated in the internship session:

**Data → Information → Insight → Decision → Action**

### Dashboard hierarchy
1. KPIs — What is happening?
2. Trends — How is it changing?
3. Drivers — Why is it changing?
4. Risk — What could go wrong?
5. Opportunity — Where can the business grow?
6. Action — What should management do?


## 1. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 2. Sample Business Data

A self-contained sample sales dataset is created so the submission contains complete analysis logic without requiring a separate dataset file.

In [ ]:
np.random.seed(42)
n = 300
dates = pd.date_range("2025-01-01", "2025-12-31", periods=n)

df = pd.DataFrame({
    "Order_ID": [f"ORD{i:04d}" for i in range(1, n+1)],
    "Order_Date": dates,
    "Region": np.random.choice(["North", "South", "East", "West"], n),
    "Category": np.random.choice(["Electronics", "Furniture", "Office Supplies", "Accessories"], n),
    "Customer_Segment": np.random.choice(["Consumer", "Corporate", "Small Business"], n),
    "Customer_ID": [f"CUST{np.random.randint(1,151):04d}" for _ in range(n)],
    "Quantity": np.random.randint(1, 8, n),
    "Unit_Price": np.random.uniform(20, 800, n).round(2)
})

df["Revenue"] = (df["Quantity"] * df["Unit_Price"]).round(2)
df["Profit"] = (df["Revenue"] * np.random.uniform(0.08, 0.30, n)).round(2)
df["Profit_Margin_%"] = (df["Profit"] / df["Revenue"] * 100).round(2)

df.head()


## 3. Data Quality Check

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## 4. Executive KPIs

The executive view focuses only on decision-useful KPIs.

In [ ]:
total_revenue = df["Revenue"].sum()
total_orders = df["Order_ID"].nunique()
total_customers = df["Customer_ID"].nunique()
avg_order_value = total_revenue / total_orders
total_profit = df["Profit"].sum()
monthly_revenue = df.groupby(df["Order_Date"].dt.to_period("M"))["Revenue"].sum()
growth_pct = (monthly_revenue.iloc[-1] / monthly_revenue.iloc[0] - 1) * 100

kpis = pd.DataFrame({
    "KPI": ["Revenue", "Orders", "Customers", "Average Order Value", "Profit", "Growth %"],
    "Value": [total_revenue, total_orders, total_customers, avg_order_value, total_profit, growth_pct]
})
kpis


## 5. Trend Analysis

In [ ]:
monthly = df.assign(Month=df["Order_Date"].dt.to_period("M").astype(str)).groupby("Month", as_index=False)["Revenue"].sum()

plt.figure(figsize=(12,5))
plt.plot(monthly["Month"], monthly["Revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Drivers: Regional and Category Performance

In [ ]:
region = df.groupby("Region").agg(Revenue=("Revenue","sum"), Profit=("Profit","sum"), Orders=("Order_ID","nunique")).sort_values("Revenue", ascending=False)
category = df.groupby("Category").agg(Revenue=("Revenue","sum"), Profit=("Profit","sum"), Orders=("Order_ID","nunique")).sort_values("Revenue", ascending=False)

print("Regional performance")
display(region)
print("Category performance")
display(category)


## 7. Customer Analysis

In [ ]:
segment = df.groupby("Customer_Segment").agg(Revenue=("Revenue","sum"), Profit=("Profit","sum"), Orders=("Order_ID","nunique"), Customers=("Customer_ID","nunique")).sort_values("Revenue", ascending=False)
segment


## 8. Risk Analysis

Potential risks are identified from below-average profit margins.

In [ ]:
avg_margin = df["Profit_Margin_%"].mean()
risk = (df.groupby(["Region","Category"]).agg(Revenue=("Revenue","sum"), Profit=("Profit","sum"), Profit_Margin=("Profit_Margin_%","mean")).reset_index())
risk = risk[risk["Profit_Margin"] < avg_margin].sort_values("Profit_Margin")
print(f"Average margin: {avg_margin:.2f}%")
risk.head(10)


## 9. Opportunity Analysis

In [ ]:
opportunity = region.copy().sort_values("Profit", ascending=False)
print("Top regions by profit:")
display(opportunity.head(3))

print("Top categories by revenue:")
display(category.head(3))


## 10. Fact → Insight → Opportunity → Action

In [ ]:
top_region = region.index[0]
top_category = category.index[0]

print("FACT")
print(f"The highest-revenue region is {top_region}, and the highest-revenue category is {top_category}.")

print("\nINSIGHT")
print("Performance is concentrated in the strongest regions and product categories.")

print("\nOPPORTUNITY")
print(f"Targeted investment and cross-selling can be considered for the strongest region ({top_region}) and category ({top_category}).")

print("\nRISK")
print("Low-margin region-category combinations should be monitored to protect profitability.")

print("\nACTION")
print("Protect high-performing segments, investigate low-margin combinations, and review KPIs monthly.")


## 11. AI Analyst Prompt

Use only verified notebook outputs when using Gemini, IBM watsonx, or another AI assistant. Do not invent numbers.

In [ ]:
ai_prompt = """You are an executive business analyst. Using only the verified KPI, trend, driver, risk and opportunity results from this notebook, provide:
1. Five key findings
2. Three risks
3. Three opportunities
4. Five recommended management actions
Do not invent or change any numerical values. Clearly separate facts from recommendations."""
print(ai_prompt)


## 12. Conclusion

This project follows the BI storytelling approach from the training: a dashboard should reduce noise and help management move from facts to decisions and actions. The analysis covers executive KPIs, trends, drivers, risks, opportunities and recommended actions.